# Assignment 3: Two Averages, One Argument

**Input file:** `teaching_prices_clean.csv` (96 rows: 4 items, 2 cities, 12 weekly dates from 1 Jan to 19 Mar 2026). Unit: PKR per kg. The file labels the data as illustrative teaching data, not actual PBS observations.

## Method

- **Mean:** sum of prices divided by the number of observations.
- **Median:** middle value of the sorted prices; the average of the two middle values when the count is even.
- **First-to-last change:** the two city prices are averaged on each date. Change = (last-date average − first-date average) ÷ first-date average × 100.
- Item-level mean and median pool both cities (24 observations per item).

## Flagged-value comparison

The flagged observation is Sweetener D, City 1, 22 January 2026. The cleaned file holds PKR 157.50 at that row. The comparison substitutes the earlier flagged value, PKR 9,999, at the same row. This value is not in the current file, so the "with flag" figures are a reconstruction.

## Output

The code cell below prints the item-level summary, the city-level summary, and the with/without comparison.

In [1]:
import pandas as pd
df = pd.read_csv("teaching_prices_clean.csv")
df["date"] = pd.to_datetime(df["date"])
summary = df.groupby("item")["price"].agg(n="count", mean="mean", median="median")
by_date = df.groupby(["item", "date"])["price"].mean().reset_index().sort_values("date")
first_last = by_date.groupby("item")["price"].agg(first="first", last="last")
first_last["change_pct"] = (first_last["last"] - first_last["first"]) / first_last["first"] * 100
print("Item-level summary (both cities pooled)")
display(summary.join(first_last).round(2))
city = df.groupby(["item", "city"]).agg(n=("price", "count"), mean=("price", "mean"), median=("price", "median"))
print("City-level summary")
display(city.round(2))
flag = (df["item"] == "Sweetener D") & (df["city"] == "City 1") & (df["date"] == "2026-01-22")
print("Cleaned value at flagged row:", df.loc[flag, "price"].iloc[0])
sweet = df[df["item"] == "Sweetener D"]["price"]
sweet_flag = df["price"].where(~flag, 9999)[df["item"] == "Sweetener D"]
comparison = pd.DataFrame({"mean": [sweet.mean(), sweet_flag.mean()], "median": [sweet.median(), sweet_flag.median()]}, index=["Cleaned value (157.50)", "With flagged value (9,999)"])
comparison.loc["Difference"] = comparison.iloc[1] - comparison.iloc[0]
print("Sweetener D, 24 observations")
display(comparison.round(2))

Item-level summary (both cities pooled)
              n    mean  median  first   last  change_pct
item                                                     
Oil B        24  324.83  324.00  301.0  347.0       15.28
Pulse C      24  205.92  206.00  203.0  210.5        3.69
Staple A     24  111.17  111.75  104.0  117.5       12.98
Sweetener D  24  166.75  166.75  152.0  181.5       19.41
City-level summary
                     n    mean  median
item        city                      
Oil B       City 1  12  321.83  321.00
            City 2  12  327.83  327.00
Pulse C     City 1  12  202.92  202.50
            City 2  12  208.92  208.50
Staple A    City 1  12  108.17  107.75
            City 2  12  114.17  113.75
Sweetener D City 1  12  163.75  163.75
            City 2  12  169.75  169.75
Cleaned value at flagged row: 157.5
Sweetener D, 24 observations
                              mean  median
Cleaned value (157.50)      166.75  166.75
With flagged value (9,999)  576.81  167.75
Differenc